# 01 — Préparation des données (membre 1)

**Responsable :** Mamadou Lamine Barry (données et qualité)

- **Section 1 — Diagnostic** : vérifications sur `df_brut`, tel qu'il est lu depuis `data/raw/`, sans aucune modification.
- **Section 2 — Nettoyage** : application des décisions D1 à D6 validées dans `docs/journal_decisions.md`, sur une copie `df`.
- **Section 3 — Variables dérivées** : parts, taux et variation annuelle (V1 à V8).
- **Section 4 — Contrôle et sauvegarde** : vérification des totaux, puis écriture de `data/processed/urgences_propre.csv`.

Le fichier brut n'est jamais modifié.

Les cellules validées seront reportées dans `analyse_eda.ipynb` (notebook final).

## 1. Diagnostic de qualité

### 1.1 Chargement et encodage

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Racine du projet, que le notebook soit lancé depuis notebooks/ ou depuis la racine
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FICHIER_RAW = RACINE / "data" / "raw" / "Donnees_cumulatives_Urgence.csv"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

# Les 3 premiers octets EF BB BF signalent un fichier UTF-8 avec BOM
debut = FICHIER_RAW.read_bytes()[:3]
print("Premiers octets :", debut, "-> BOM UTF-8 :", debut == b"\xef\xbb\xbf")

# Contre-exemple : avec un mauvais encodage (latin-1), le BOM et les accents sont corrompus
mauvais = pd.read_csv(FICHIER_RAW, encoding="latin-1")
print("latin-1   -> 1re colonne :", repr(mauvais.columns[0]), "| région :", repr(mauvais.loc[8, "region"]))

# Lecture retenue : utf-8-sig (UTF-8 en retirant le BOM)
df_brut = pd.read_csv(FICHIER_RAW, encoding="utf-8-sig")
print("utf-8-sig -> 1re colonne :", repr(df_brut.columns[0]), "| région :", repr(df_brut.loc[8, "region"]))

Premiers octets : b'\xef\xbb\xbf' -> BOM UTF-8 : True
latin-1   -> 1re colonne : 'ï»¿annee' | région : 'ChaudiÃ¨re-Appalaches'
utf-8-sig -> 1re colonne : 'annee' | région : 'Chaudière-Appalaches'


**Ce qu'on observe.** Le fichier commence par les octets `EF BB BF` : c'est un **BOM** (*byte order mark*), qui indique un fichier encodé en **UTF-8**.

- Lu en `latin-1`, le BOM devient des caractères parasites collés au nom de la première colonne (`ï»¿annee`), et les accents sont corrompus (`ChaudiÃ¨re-Appalaches`).
- Lu en `utf-8-sig`, la première colonne s'appelle bien `annee` et les accents sont corrects.

**Décision de lecture :** `encoding="utf-8-sig"`. Cette lecture ne modifie pas les données : elle les décode correctement.

### 1.2 Dimensions

In [2]:
print(f"{df_brut.shape[0]} lignes × {df_brut.shape[1]} colonnes")
df_brut.head()

949 lignes × 27 colonnes


,annee,cumul_periode,rss,region,nom_etablissement,nom_installation,no_permis_installation,nb_visites_total,nb_usagers_75ans_et_plus_total,nb_usagers_sante_mentale_total,dms_total,nb_usagers_pec_total,delai_pec_total,nb_visites_ambulatoire,nb_usagers_75ans_et_plus_amb,nb_usagers_sante_mentale_amb,dms_ambulatoire,nb_usagers_pec_ambulatoire,delai_pec_ambulatoire,nb_visites_sur_civiere,nb_usagers_sur_civiere_plus_24h,nb_usagers_sur_civiere_plus_48h,dms_sur_civiere,nb_usagers_pec_sur_civiere,delai_pec_sur_civiere,nb_usagers_75ans_et_plus_civ,nb_usagers_sante_mentale_civ
0,2022-2023,7,99,Total provincial,Total provincial,Total provincial,.,1930100,287058,110038,8.878886,1530330,2.812493,1333437,107998,45108,4.889529,940159,3.342296,596663,140469,42656,17.794400,590171,1.968501,179060,64930
1,2023-2024,7,99,Total provincial,Total provincial,Total provincial,.,1884897,303633,110497,8.758813,1523003,2.793726,1270039,117571,43685,4.882417,914766,3.305984,614858,135394,34527,16.765825,608237,2.023308,186062,66812
2,2026-2027,7,99,Total provincial,Total provincial,Total provincial,.,1858526,327606,109963,9.044367,1475080,3.041923,1244278,132358,42854,5.053704,868126,3.522228,614248,136437,32375,17.128227,606954,2.354944,195248,67109
3,2021-2022,7,99,Total provincial,Total provincial,Total provincial,.,1812368,266999,110115,8.304020,1475267,2.519820,1229233,98796,45913,4.580041,897384,3.000874,583135,121839,29852,16.154068,577883,1.772800,168203,64202
4,2025-2026,7,99,Total provincial,Total provincial,Total provincial,.,2011715,347228,120027,8.596694,1624773,2.802467,1335576,134799,46271,4.782049,956254,3.287301,676139,136959,31334,16.131754,668519,2.108957,212429,73756


**Ce qu'on observe.** Le fichier contient **949 lignes et 27 colonnes**.

- Les premières lignes ne sont pas des urgences mais le **« Total provincial »** de chaque année, avec un numéro de permis `.`. Le fichier mélange donc plusieurs niveaux d'agrégation (vérifié en 1.6).
- Les lignes ne sont pas triées par année (2022-2023, puis 2023-2024, puis 2026-2027…).
- Une ligne correspond à **une entité (installation, établissement, région ou province) pour une année financière**, cumulée du 1er avril jusqu'à la période indiquée dans `cumul_periode`.

### 1.3 Types des colonnes

In [3]:
types = pd.DataFrame({
    "type": df_brut.dtypes.astype(str),
    "exemple (1re ligne)": df_brut.iloc[0],
    "valeurs distinctes": df_brut.nunique(),
})
print(types["type"].value_counts().to_string())
types

type
int64      16
float64     6
str         5


,type,exemple (1re ligne),valeurs distinctes
annee,str,2022-2023,6
cumul_periode,int64,7,1
rss,int64,99,17
region,str,Total provincial,17
nom_etablissement,str,Total provincial,31
nom_installation,str,Total provincial,123
no_permis_installation,str,.,121
nb_visites_total,int64,1930100,831
nb_usagers_75ans_et_plus_total,int64,287058,783
nb_usagers_sante_mentale_total,int64,110038,706


**Ce qu'on observe.** Les types lus par pandas sont cohérents avec le dictionnaire du MSSS :

- **5 colonnes texte** (`str`) : `annee`, `region`, `nom_etablissement`, `nom_installation`, `no_permis_installation`.
- **16 colonnes entières** (`int64`) : les 14 comptes (visites, usagers de 75 ans et plus, santé mentale, prises en charge, séjours sur civière de plus de 24 h et de plus de 48 h), plus les codes `cumul_periode` et `rss`.
- **6 colonnes décimales** (`float64`) : les durées moyennes de séjour (`dms_*`) et les délais de prise en charge (`delai_pec_*`).

**Points d'attention :**
- `annee` est une **année financière** (`"2021-2022"`, du 1er avril au 31 mars), et non une date. On la garde en texte, ce qui suffit pour trier et grouper.
- `no_permis_installation` est lu en texte à cause du marqueur `"."` utilisé dans les lignes de total. C'est un identifiant, pas une quantité, donc le texte convient.
- `rss` est un code de région : un nombre qu'on ne doit pas additionner ni moyenner.
- `cumul_periode` n'a **qu'une seule valeur (7)** dans tout le fichier (voir 1.8).
- Le dictionnaire ne précise pas l'unité des durées (« nombres décimaux »). Les seuils de 24 h et 48 h et les valeurs observées (environ 7 à 18) indiquent des **heures** : c'est une hypothèse à signaler dans le rapport.

### 1.4 Valeurs manquantes par colonne

In [4]:
# Valeurs manquantes au sens de pandas (NaN) et marqueur texte "." utilisé par la source
manquants = pd.DataFrame({
    "NaN": df_brut.isna().sum(),
    "marqueur '.'": df_brut.apply(lambda s: s.astype(str).str.strip().eq(".").sum()),
})
manquants["% des lignes"] = (manquants.sum(axis=1) / len(df_brut) * 100).round(1)
print("Colonnes sans aucune valeur manquante :", (manquants[["NaN", "marqueur '.'"]].sum(axis=1) == 0).sum(), "sur", len(manquants))
display(manquants[manquants[["NaN", "marqueur '.'"]].sum(axis=1) > 0])

# Détail des lignes contenant un NaN
df_brut.loc[df_brut.isna().any(axis=1),
            ["annee", "region", "nom_installation", "nb_visites_total", "nb_visites_sur_civiere",
             "dms_sur_civiere", "delai_pec_sur_civiere"]]

Colonnes sans aucune valeur manquante : 24 sur 27


,NaN,marqueur '.',% des lignes
no_permis_installation,0,265,27.9
dms_sur_civiere,1,0,0.1
delai_pec_sur_civiere,1,0,0.1


,annee,region,nom_installation,nb_visites_total,nb_visites_sur_civiere,dms_sur_civiere,delai_pec_sur_civiere
424,2026-2027,Error,CENTRE DE SANTÉ DE RADISSON,400,0,NaN,NaN


**Ce qu'on observe.** **24 colonnes sur 27** n'ont aucune valeur manquante. Les 3 autres :

- `no_permis_installation` contient **265 fois le marqueur `"."`** (27,9 % des lignes). pandas ne le considère pas comme manquant, car c'est du texte. Ces 265 lignes sont exactement les lignes de total (vérifié en 1.6) : une ligne agrégée n'a pas de numéro de permis. Ce n'est donc pas une vraie donnée manquante.
- `dms_sur_civiere` et `delai_pec_sur_civiere` ont **1 seul NaN chacun**, sur la même ligne : le **Centre de santé de Radisson** en 2026-2027. Cette installation n'a eu **aucune visite sur civière** (`nb_visites_sur_civiere = 0`) : une moyenne calculée sur 0 patient n'existe pas. **Le vide est logique et ne doit pas être remplacé** par 0 ou par une moyenne, ce qui fausserait les durées.

On remarque aussi que cette ligne porte la région `Error` (voir 1.7).

### 1.5 Doublons

In [5]:
print("Doublons complets (toutes colonnes identiques) :", df_brut.duplicated().sum())

cle = ["annee", "rss", "nom_etablissement", "nom_installation"]
print(f"Doublons sur la clé {cle} :", df_brut.duplicated(cle).sum())

# Parmi les lignes qui ont un vrai numéro de permis : une installation par année ?
avec_permis = df_brut[df_brut["no_permis_installation"] != "."]
print("Doublons (année, numéro de permis) :", avec_permis.duplicated(["annee", "no_permis_installation"]).sum())
print("Doublons (année, nom d'installation) :", avec_permis.duplicated(["annee", "nom_installation"]).sum())

Doublons complets (toutes colonnes identiques) : 0


Doublons sur la clé ['annee', 'rss', 'nom_etablissement', 'nom_installation'] : 0
Doublons (année, numéro de permis) : 0
Doublons (année, nom d'installation) : 0


**Ce qu'on observe.** Le fichier ne contient **aucun doublon** :

- aucune ligne n'est répétée à l'identique ;
- la combinaison (année, région, établissement, installation) est unique ;
- parmi les lignes qui ont un vrai numéro de permis, chaque installation n'apparaît **qu'une fois par année**, que l'on vérifie par numéro de permis ou par nom.

La clé **(année, numéro de permis)** identifie donc chaque urgence de façon unique. On l'utilisera pour suivre une installation d'une année à l'autre. Aucune suppression de doublon n'est nécessaire.

### 1.6 Lignes de total

In [6]:
# Les lignes agrégées contiennent « Total » dans le nom de l'installation
est_total = df_brut["nom_installation"].str.contains("total", case=False)
niveau = df_brut["nom_installation"].where(est_total, "Installation (détail)")
print(niveau.value_counts().to_string(), "\n")

# Le marqueur "." du numéro de permis correspond-il exactement aux lignes de total ?
print("Permis '.' <=> ligne de total :", (df_brut["no_permis_installation"].eq(".") == est_total).all(), "\n")

# Contrôle : la somme des installations redonne-t-elle le total provincial ?
detail = df_brut[~est_total]
controle = pd.DataFrame({
    "total_provincial": df_brut[niveau == "Total provincial"].set_index("annee")["nb_visites_total"],
    "somme_installations": detail.groupby("annee")["nb_visites_total"].sum(),
    "somme_toutes_lignes": df_brut.groupby("annee")["nb_visites_total"].sum(),
}).sort_index()
controle["écart_installations"] = controle["total_provincial"] - controle["somme_installations"]
controle

nom_installation
Installation (détail)    684
Total établissement      168
Total régional            91
Total provincial           6 

Permis '.' <=> ligne de total : True 



,total_provincial,somme_installations,somme_toutes_lignes,écart_installations
annee,,,,
2021-2022,1812368,1812368,7249472,0
2022-2023,1930100,1930100,7720400,0
2023-2024,1884897,1884897,7539588,0
2024-2025,2077882,2077882,8311528,0
2025-2026,2011715,2011715,8046860,0
2026-2027,1858526,1858526,7434104,0


**Ce qu'on observe.** Le fichier empile **4 niveaux d'agrégation** :

| Niveau | Lignes |
|---|---|
| Installation (détail) | 684 |
| Total établissement | 168 |
| Total régional | 91 |
| Total provincial | 6 (1 par année) |

- Le marqueur `"."` du numéro de permis correspond **exactement** aux lignes de total. On peut donc les repérer par l'un ou l'autre critère.
- Pour chaque année, la **somme des 684 installations est exactement égale au total provincial** (écart de 0). Les lignes de détail sont donc complètes.
- En revanche, si on additionne **toutes** les lignes sans filtrer, on obtient **4 fois** le total réel. Par exemple, 7 249 472 visites au lieu de 1 812 368 en 2021-2022, car chaque visite est comptée au niveau de l'installation, de l'établissement, de la région et de la province.

**Conséquence :** toute analyse doit porter **soit** sur les installations, **soit** sur un seul niveau de total, jamais sur un mélange. C'est la décision n° 1 de `docs/journal_decisions.md` (à appliquer en section 2).

### 1.7 Cohérence des libellés et des sommes

In [7]:
# Correspondance entre code RSS et nom de région
display(df_brut.groupby(["rss", "region"]).size().rename("lignes").reset_index())

# Règles logiques, vérifiées sur les lignes d'installation
regles = {
    "visites totales = ambulatoires + civière": (detail["nb_visites_total"] == detail["nb_visites_ambulatoire"] + detail["nb_visites_sur_civiere"]),
    "civière > 48 h <= civière > 24 h": (detail["nb_usagers_sur_civiere_plus_48h"] <= detail["nb_usagers_sur_civiere_plus_24h"]),
    "civière > 24 h <= visites sur civière": (detail["nb_usagers_sur_civiere_plus_24h"] <= detail["nb_visites_sur_civiere"]),
    "75 ans et plus <= visites totales": (detail["nb_usagers_75ans_et_plus_total"] <= detail["nb_visites_total"]),
    "pris en charge <= visites totales": (detail["nb_usagers_pec_total"] <= detail["nb_visites_total"]),
    "aucune valeur négative": (detail.select_dtypes("number").fillna(0) >= 0).all(axis=1),
}
pd.DataFrame({"lignes conformes": {k: int(v.sum()) for k, v in regles.items()},
              "lignes non conformes": {k: int((~v).sum()) for k, v in regles.items()}})

,rss,region,lignes
0,1,Bas-Saint-Laurent,60
1,2,Saguenay-Lac-Saint-Jean,48
2,3,Capitale-Nationale,96
3,4,Mauricie et Centre-du-Québec,50
4,5,Estrie,60
5,6,Montréal,181
6,7,Outaouais,60
7,8,Abitibi-Témiscamingue,48
8,9,Côte-Nord,60
9,10,Error,10


,lignes conformes,lignes non conformes
visites totales = ambulatoires + civière,684,0
civière > 48 h <= civière > 24 h,684,0
civière > 24 h <= visites sur civière,684,0
75 ans et plus <= visites totales,684,0
pris en charge <= visites totales,684,0
aucune valeur négative,684,0


**Ce qu'on observe.**

**Libellés des régions :** chaque code RSS correspond à un seul nom de région, avec une exception : le code **RSS 10 porte le nom `Error`** sur 10 lignes. Ce code correspond au **Nord-du-Québec**, d'après la nomenclature des régions sociosanitaires et le seul établissement concerné, Santé Québec Baie-James. C'est une erreur d'étiquette de la source, à corriger en section 2 (décision n° 2). La région 99 est le total provincial.

**Règles logiques :** les 6 règles vérifiées sont respectées par **les 684 lignes d'installation** :
- les visites totales sont exactement la somme des visites ambulatoires et sur civière ;
- les séjours de plus de 48 h ne dépassent jamais ceux de plus de 24 h, qui ne dépassent jamais le nombre de visites sur civière ;
- aucune sous-catégorie ne dépasse le total ;
- aucune valeur n'est négative.

Les données numériques sont donc **cohérentes en interne** : on n'a pas besoin de les corriger.

### 1.8 Années partielles et périmètre par année

In [8]:
# Toutes les années couvrent-elles la même période (cumul_periode) ?
display(pd.crosstab(detail["annee"], detail["cumul_periode"]))

# Nombre d'années de présence de chaque installation
annees = sorted(detail["annee"].unique())
presence = detail.groupby("no_permis_installation")["annee"].nunique()
print("Installations selon le nombre d'années présentes :", presence.value_counts().sort_index().to_dict(), "\n")

constantes = presence[presence == len(annees)].index
par_annee = detail.groupby("annee").agg(installations=("no_permis_installation", "nunique"),
                                       visites=("nb_visites_total", "sum"))
par_annee["visites_périmètre_constant"] = (detail[detail["no_permis_installation"].isin(constantes)]
                                          .groupby("annee")["nb_visites_total"].sum())
par_annee

cumul_periode,7
annee,
2021-2022,115
2022-2023,115
2023-2024,115
2024-2025,115
2025-2026,116
2026-2027,108


Installations selon le nombre d'années présentes : {1: 4, 2: 1, 5: 12, 6: 103} 



,installations,visites,visites_périmètre_constant
annee,,,
2021-2022,115,1812368,1612452
2022-2023,115,1930100,1723274
2023-2024,115,1884897,1682597
2024-2025,115,2077882,1853830
2025-2026,116,2011715,1794352
2026-2027,108,1858526,1844544


**Ce qu'on observe.**

**Pas d'année partielle au sens de la période couverte :** `cumul_periode` vaut **7 pour toutes les années**. Chaque année financière couvre la même fenêtre, du 1er avril à la fin de la 7e période financière (environ la mi-octobre). Les années sont donc comparables entre elles, même la plus récente (2026-2027). En revanche, ce ne sont **pas des années complètes** : il manque les périodes 8 à 13, qui comprennent l'hiver. Cela limite l'interprétation et doit être mentionné.

**Mais le périmètre change d'une année à l'autre :**
- **103 installations** sont présentes les 6 années ;
- 12 installations sont présentes 5 ans, 1 installation l'est 2 ans et 4 ne le sont qu'1 an ;
- le nombre d'installations passe de **115** (2021-2025) à **116** (2025-2026), puis à **108** (2026-2027).

**Effet sur les totaux :** le total des visites **baisse de 7,6 %** entre 2025-2026 et 2026-2027 (de 2 011 715 à 1 858 526). Pourtant, à **périmètre constant** (les 103 installations présentes partout), les visites **augmentent de 2,8 %** (de 1 794 352 à 1 844 544). La baisse apparente est donc un **effet de composition** dû aux installations manquantes, et non une baisse de fréquentation. Pour comparer les années, il faudra se limiter aux 103 installations (décision n° 3).

In [9]:
# Installations présentes l'avant-dernière année mais absentes la dernière
avant, derniere = annees[-2], annees[-1]
permis_avant = set(detail.loc[detail["annee"] == avant, "no_permis_installation"])
permis_derniere = set(detail.loc[detail["annee"] == derniere, "no_permis_installation"])
absentes = detail[(detail["annee"] == avant) & detail["no_permis_installation"].isin(permis_avant - permis_derniere)]
visites_absentes = f"{absentes['nb_visites_total'].sum():,}".replace(",", " ")
print(f"{len(absentes)} installations présentes en {avant} et absentes en {derniere}, "
      f"soit {visites_absentes} visites en {avant}")
absentes[["region", "nom_installation", "nb_visites_total"]].sort_values("region")

12 installations présentes en 2025-2026 et absentes en 2026-2027, soit 216 385 visites en 2025-2026


,region,nom_installation,nb_visites_total
298,Mauricie et Centre-du-Québec,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES SOCIAUX DE FORTIERVILLE,4462
315,Mauricie et Centre-du-Québec,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES SOCIAUX AVELLIN-DALCOURT,12393
402,Mauricie et Centre-du-Québec,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES SOCIAUX CHRIST-ROI,12765
519,Mauricie et Centre-du-Québec,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES SOCIAUX DU HAUT-SAINT-MAURICE,6800
535,Mauricie et Centre-du-Québec,HÔPITAL DU CENTRE-DE-LA-MAURICIE,19585
562,Mauricie et Centre-du-Québec,HÔTEL-DIEU D'ARTHABASKA,21676
761,Mauricie et Centre-du-Québec,PAVILLON SAINTE-MARIE,33493
780,Mauricie et Centre-du-Québec,HÔPITAL SAINTE-CROIX,24379
265,Montréal,HÔPITAL DU SACRÉ-COEUR-DE-MONTRÉAL,37977
658,Montréal,HÔPITAL JEAN-TALON,23269


**Ce qu'on observe.** **12 installations** présentes en 2025-2026 sont absentes en 2026-2027. Elles représentaient **216 385 visites** sur la période :

- **8 en Mauricie et Centre-du-Québec** : c'est **toute la région**, qui n'a plus aucune installation en 2026-2027. On y trouve les hôpitaux de Trois-Rivières (Pavillon Sainte-Marie), Drummondville (Sainte-Croix), Victoriaville (Hôtel-Dieu d'Arthabaska) et Shawinigan (Centre-de-la-Mauricie) ;
- **4 à Montréal** : Sacré-Cœur, Jean-Talon, Fleury et Albert-Prévost.

Ces absences sont regroupées dans deux régions précises. Cela évoque un **problème de transmission ou de réorganisation administrative** des données (par exemple un changement d'établissement responsable), plutôt qu'une fermeture réelle de ces urgences. Le fichier ne permet pas de le confirmer : c'est une **limite** à mentionner. Elle empêche aussi toute analyse de la Mauricie et Centre-du-Québec en 2026-2027 et fausse la comparaison de Montréal avec les années précédentes.

À l'inverse, 5 installations du Nord-du-Québec (Baie-James) n'apparaissent qu'en 2025-2026 ou 2026-2027.

### 1.9 Valeurs atypiques (règle de 1,5 × l'écart interquartile)

In [10]:
indicateurs = ["nb_visites_total", "dms_total", "dms_sur_civiere", "delai_pec_total",
               "nb_usagers_sur_civiere_plus_24h"]

resume, exemples = [], {}
for col in indicateurs:
    q1, q3 = detail[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    hors = detail[(detail[col] < bas) | (detail[col] > haut)]
    resume.append({"indicateur": col, "médiane": detail[col].median(), "borne haute": haut,
                   "max": detail[col].max(), "lignes atypiques": len(hors),
                   "installations concernées": hors["nom_installation"].nunique()})
    exemples[col] = hors["nom_installation"].value_counts().head(3)

display(pd.DataFrame(resume).round(2))
for col, serie in exemples.items():
    print(f"\n{col} — installations le plus souvent atypiques (nombre d'années) :")
    print(serie.to_string())

,indicateur,médiane,borne haute,max,lignes atypiques,installations concernées
0,nb_visites_total,13009.50,44426.75,55314.00,17,5
1,dms_total,6.91,17.57,23.69,8,3
2,dms_sur_civiere,13.12,32.90,38.41,5,2
3,delai_pec_total,2.43,5.45,7.58,11,6
4,nb_usagers_sur_civiere_plus_24h,472.00,4529.38,6233.00,34,11



nb_visites_total — installations le plus souvent atypiques (nombre d'années) :
nom_installation
HÔPITAL DE LA CITÉ-DE-LA-SANTÉ    6
HÔPITAL GÉNÉRAL JUIF              6
CHU SAINTE-JUSTINE                2

dms_total — installations le plus souvent atypiques (nombre d'années) :
nom_installation
HÔPITAL EN SANTÉ MENTALE ALBERT-PRÉVOST                5
INSTITUT UNIVERSITAIRE EN SANTÉ MENTALE DE MONTRÉAL    2
HÔPITAL ANNA-LABERGE                                   1

dms_sur_civiere — installations le plus souvent atypiques (nombre d'années) :
nom_installation
HÔPITAL EN SANTÉ MENTALE ALBERT-PRÉVOST    4
HÔPITAL ANNA-LABERGE                       1

delai_pec_total — installations le plus souvent atypiques (nombre d'années) :
nom_installation
HÔPITAL DE GATINEAU          3
HÔPITAL DE HULL              3
HÔPITAL PIERRE-LE GARDEUR    2

nb_usagers_sur_civiere_plus_24h — installations le plus souvent atypiques (nombre d'années) :
nom_installation
HÔPITAL DE SAINT-JÉRÔME                        

**Ce qu'on observe.** La règle de 1,5 × l'écart interquartile repère des valeurs atypiques pour chaque indicateur. Ce sont toutes des **valeurs plausibles**, liées au type d'installation, et **non des erreurs de saisie** :

- **Nombre de visites** (17 lignes, 5 installations) : les grandes urgences, en particulier la **Cité-de-la-Santé** (Laval) et l'**Hôpital général juif**, atypiques les 6 années, avec jusqu'à 55 314 visites contre une médiane de 13 010. C'est un effet de **taille**.
- **Durée moyenne de séjour totale** (8 lignes, 3 installations) et **sur civière** (5 lignes, 2 installations) : surtout l'**Hôpital en santé mentale Albert-Prévost**, avec jusqu'à 23,7 h de séjour total et 38,4 h sur civière. L'Institut universitaire en santé mentale de Montréal apparaît aussi. Ces urgences **psychiatriques** ont une mission différente des urgences générales.
- **Délai de prise en charge** (11 lignes, 6 installations) : les hôpitaux de **Gatineau** et de **Hull** (Outaouais), avec jusqu'à 7,6 h contre une médiane de 2,4 h. C'est un signal **régional** intéressant pour l'analyse.
- **Séjours sur civière de plus de 24 h** (34 lignes, 11 installations) : de grands hôpitaux comme **Saint-Jérôme**, le **CHUM** et **Maisonneuve-Rosemont**. Comme c'est un **nombre absolu**, il dépend surtout de la taille de l'urgence. Il faudra le transformer en **taux**, en divisant par le nombre de visites sur civière, pour comparer les installations entre elles.

**Conséquence :** on **garde** ces valeurs, car elles sont réelles, et on les commente. Les urgences psychiatriques (Albert-Prévost, Institut universitaire en santé mentale) pourront servir à un **contrôle complémentaire** : refaire un résultat avec et sans elles pour mesurer leur influence.

### 1.10 Synthèse du diagnostic

| Aspect | Résultat | Action (appliquée en section 2) |
|---|---|---|
| Encodage | UTF-8 avec BOM | Lire avec `encoding="utf-8-sig"` (déjà fait) |
| Dimensions | 949 lignes × 27 colonnes | — |
| Types | Cohérents ; `annee` = année financière en texte | Aucune conversion obligatoire |
| Valeurs manquantes | 265 `"."` (lignes de total) + 1 NaN logique (Radisson, 0 visite sur civière) | Ne rien imputer |
| Doublons | Aucun ; clé unique = (année, numéro de permis) | Aucune |
| Lignes de total | 265 lignes sur 4 niveaux ; somme des installations = total provincial | Garder les 684 installations |
| Libellés | RSS 10 étiqueté `Error` (10 lignes) | Renommer en « Nord-du-Québec » |
| Cohérence interne | 6 règles sur 6 respectées | Aucune |
| Années | Même fenêtre (périodes 1 à 7) chaque année, mais périmètre variable (115 → 108 installations ; Mauricie et Centre-du-Québec absente en 2026-2027) | Comparer les années sur les 103 installations constantes |
| Valeurs atypiques | Réelles (taille, mission psychiatrique, Outaouais) | Garder, commenter et tester leur influence |

Le jeu est **de bonne qualité** : pas de doublon, presque aucune valeur manquante, des sommes cohérentes. Les **vrais risques** sont structurels : le mélange des niveaux d'agrégation, les installations manquantes en 2026-2027 et une étiquette de région erronée. Les décisions correspondantes sont décrites dans `docs/journal_decisions.md`.

## 2. Nettoyage (décisions D1 à D6)

Décisions validées par Mamadou Lamine Barry au nom de l'équipe, le 2026-11-11. Leur justification chiffrée et les pages du PDF du MSSS sont dans `docs/journal_decisions.md`. Le nettoyage travaille sur une copie `df` : `df_brut` reste intact pour les contrôles de la section 4.

### 2.1 D1 — Exclure les lignes de total

In [11]:
est_total_brut = df_brut["no_permis_installation"].eq(".")
df = df_brut.loc[~est_total_brut].copy()

print(f"{len(df_brut)} lignes -> {len(df)} lignes ({est_total_brut.sum()} lignes de total retirées)")
print("Reste-t-il un libellé « Total » ?", df["nom_installation"].str.contains("total", case=False).any())

949 lignes -> 684 lignes (265 lignes de total retirées)
Reste-t-il un libellé « Total » ? False


**Ce qu'on observe.** Les **265 lignes de total** (permis `"."`) sont retirées : il reste **684 lignes**, une par installation et par année. Aucun libellé « Total » ne subsiste.

Chaque visite n'est donc plus comptée qu'une seule fois. Les totaux régionaux et provinciaux pourront être recalculés à partir des installations (contrôle en 4.1).

### 2.2 D2 — Renommer la région « Error » en Nord-du-Québec

In [12]:
masque_error = df["region"].eq("Error")
print("Lignes « Error » :", masque_error.sum(), "| codes RSS concernés :", df.loc[masque_error, "rss"].unique().tolist())

# Garde-fou : on ne renomme que si toutes ces lignes ont bien le code RSS 10
assert (df.loc[masque_error, "rss"] == 10).all(), "Une ligne « Error » n'a pas le code RSS 10"
df.loc[masque_error, "region"] = "Nord-du-Québec"

# Contrôle : chaque code RSS correspond maintenant à un seul nom de région
regions = df.groupby("rss")["region"].agg(["nunique", "first"])
print("Codes RSS avec plusieurs noms :", (regions["nunique"] > 1).sum())
regions["first"].rename("region").to_frame().T

Lignes « Error » : 6 | codes RSS concernés : [10]
Codes RSS avec plusieurs noms : 0


rss,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
region,Bas-Saint-Laurent,Saguenay-Lac-Saint-Jean,Capitale-Nationale,Mauricie et Centre-du-Québec,Estrie,Montréal,Outaouais,Abitibi-Témiscamingue,Côte-Nord,Nord-du-Québec,Gaspésie-Îles-de-la-Madeleine,Chaudière-Appalaches,Laval,Lanaudière,Laurentides,Montérégie


**Ce qu'on observe.** Après D1, il reste **6 lignes d'installation** étiquetées `Error`. Toutes ont le code **RSS 10** : le garde-fou `assert` le vérifie avant toute modification. Elles sont renommées « Nord-du-Québec ».

Chacun des 16 codes RSS correspond maintenant à **un seul nom de région**, de 1 (Bas-Saint-Laurent) à 16 (Montérégie). Source de la correspondance RSS 10 = Nord-du-Québec : jeu MSSS *Estimations et projections de population comparables* (voir `docs/journal_decisions.md`, D2).

### 2.3 D3 — Marquer le périmètre constant (toutes les années sont gardées)

In [13]:
nb_annees = df["annee"].nunique()
annees_presentes = df.groupby("no_permis_installation")["annee"].transform("nunique")
df["perimetre_constant"] = annees_presentes.eq(nb_annees)

print("Années dans le fichier :", nb_annees)
print("Installations à périmètre constant :", df.loc[df["perimetre_constant"], "no_permis_installation"].nunique())
pd.crosstab(df["annee"], df["perimetre_constant"], margins=True, margins_name="Total")

Années dans le fichier : 6
Installations à périmètre constant : 103


perimetre_constant,False,True,Total
annee,,,
2021-2022,12,103,115
2022-2023,12,103,115
2023-2024,12,103,115
2024-2025,12,103,115
2025-2026,13,103,116
2026-2027,5,103,108
Total,66,618,684


**Ce qu'on observe.** Toutes les années sont gardées, et une colonne booléenne `perimetre_constant` est ajoutée. Elle vaut `True` si l'installation (identifiée par son numéro de permis) est présente les **6 années**.

- **103 installations** sont à périmètre constant, soit **618 lignes** (103 × 6), présentes chaque année.
- **66 lignes** sont hors périmètre constant : 12 par année de 2021-2022 à 2024-2025 (les installations absentes en 2026-2027), 13 en 2025-2026 (les mêmes, plus Chibougamau, qui apparaît cette année-là) et 5 en 2026-2027 (Chibougamau et les 4 nouvelles installations de Baie-James).

**Mode d'emploi pour l'équipe :** pour comparer des années entre elles, filtrer `df[df["perimetre_constant"]]`. Pour décrire une seule année, toutes les lignes de cette année peuvent servir.

### 2.4 D4, D5, D6 — Ce qui est volontairement conservé

In [14]:
# D4 : la seule valeur manquante reste vide (aucune imputation)
print("NaN par colonne :", df.isna().sum()[df.isna().sum() > 0].to_dict())
print("Visites sur civière sur la ligne concernée :",
      df.loc[df["dms_sur_civiere"].isna(), ["annee", "nom_installation", "nb_visites_sur_civiere"]].to_dict("records"))

# D5 : aucune valeur atypique retirée ; D6 : durées gardées telles quelles (heures, hypothèse)
colonnes_duree = [c for c in df.columns if c.startswith(("dms_", "delai_pec_"))]
print("Colonnes de durée (en heures, hypothèse D6) :", colonnes_duree)

# Ordre de lecture stable : année, région, établissement, installation
df = df.sort_values(["annee", "rss", "nom_etablissement", "nom_installation"]).reset_index(drop=True)
print("\nDimensions finales :", df.shape)

NaN par colonne : {'dms_sur_civiere': 1, 'delai_pec_sur_civiere': 1}
Visites sur civière sur la ligne concernée : [{'annee': '2026-2027', 'nom_installation': 'CENTRE DE SANTÉ DE RADISSON', 'nb_visites_sur_civiere': 0}]
Colonnes de durée (en heures, hypothèse D6) : ['dms_total', 'delai_pec_total', 'dms_ambulatoire', 'delai_pec_ambulatoire', 'dms_sur_civiere', 'delai_pec_sur_civiere']

Dimensions finales : (684, 28)


**Ce qu'on observe.**

- **D4** : les 2 seules valeurs manquantes (`dms_sur_civiere` et `delai_pec_sur_civiere`) sont sur la même ligne : Radisson, en 2026-2027, avec 0 visite sur civière. Elles restent vides.
- **D5** : aucune ligne n'a été retirée pour valeur atypique.
- **D6** : les 6 colonnes de durée sont gardées telles quelles. Elles sont interprétées comme des **heures** ; c'est une hypothèse, car le PDF du MSSS ne précise pas l'unité (p. 3 et 4).

Les lignes sont triées par année, région, établissement et installation. Le tableau final compte **684 lignes × 28 colonnes** : les 27 d'origine plus `perimetre_constant`.

## 3. Variables dérivées

Les comptes bruts dépendent surtout de la **taille** de l'urgence : une grande urgence a forcément plus de patients de 75 ans et plus ou plus de séjours de plus de 24 h. Pour comparer des installations de tailles différentes, on construit des **parts et des taux** (en %), ainsi qu'une **variation annuelle**. Ces variables sont documentées dans `docs/journal_decisions.md` (V1 à V8).

### 3.1 Année numérique, parts et taux

In [15]:
# V1 — Année de début de l'année financière (« 2021-2022 » -> 2021), pour trier et calculer des écarts d'années
df["annee_debut"] = df["annee"].str[:4].astype(int)


def part_pct(numerateur, denominateur):
    """Part en % ; NaN si le dénominateur vaut 0 (part non définie, pas de division par zéro)."""
    return 100 * numerateur / denominateur.where(denominateur > 0)


# Parts calculées sur l'ensemble des visites
df["part_75ans_plus_pct"] = part_pct(df["nb_usagers_75ans_et_plus_total"], df["nb_visites_total"])      # V2
df["part_sante_mentale_pct"] = part_pct(df["nb_usagers_sante_mentale_total"], df["nb_visites_total"])   # V3
df["part_civiere_pct"] = part_pct(df["nb_visites_sur_civiere"], df["nb_visites_total"])                # V4
df["taux_pec_pct"] = part_pct(df["nb_usagers_pec_total"], df["nb_visites_total"])                      # V5

# Taux calculés sur les seules visites sur civière
df["taux_civiere_24h_pct"] = part_pct(df["nb_usagers_sur_civiere_plus_24h"], df["nb_visites_sur_civiere"])  # V6
df["taux_civiere_48h_pct"] = part_pct(df["nb_usagers_sur_civiere_plus_48h"], df["nb_visites_sur_civiere"])  # V7

parts = ["part_75ans_plus_pct", "part_sante_mentale_pct", "part_civiere_pct", "taux_pec_pct",
         "taux_civiere_24h_pct", "taux_civiere_48h_pct"]
df[["annee", "nom_installation", "nb_visites_total"] + parts].head(3)

,annee,nom_installation,nb_visites_total,part_75ans_plus_pct,part_sante_mentale_pct,part_civiere_pct,taux_pec_pct,taux_civiere_24h_pct,taux_civiere_48h_pct
0,2021-2022,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,10418,17.037819,5.010559,28.825110,83.682089,2.297702,0.000000
1,2021-2022,CENTRE MULTISERVICES DE SANTÉ ET DE SERVICES SOCIAUX DE TROIS-PISTOLES,5002,15.893643,3.198721,12.734906,90.563774,4.395604,0.313972
2,2021-2022,CLSC DE POHÉNÉGAMOOK,4519,12.480637,3.053773,5.443682,89.090507,0.406504,0.000000


**Ce qu'on observe.** Sept nouvelles colonnes sont créées. Les parts et les taux sont en **pourcentage (0 à 100)** :

| ID | Variable | Calcul | Sens |
|---|---|---|---|
| V1 | `annee_debut` | 4 premiers caractères de `annee` | « 2021-2022 » → 2021 |
| V2 | `part_75ans_plus_pct` | 75 ans et plus / visites totales | Poids des patients âgés |
| V3 | `part_sante_mentale_pct` | santé mentale / visites totales | Poids des problématiques de santé mentale |
| V4 | `part_civiere_pct` | visites sur civière / visites totales | Poids des cas assez lourds pour une civière |
| V5 | `taux_pec_pct` | usagers pris en charge / visites totales | Part des visites vues par un médecin (exclut les réorientés et les départs avant prise en charge, PDF p. 3) |
| V6 | `taux_civiere_24h_pct` | séjours sur civière ≥ 24 h / visites sur civière | Part des séjours sur civière de 24 h ou plus |
| V7 | `taux_civiere_48h_pct` | séjours sur civière ≥ 48 h / visites sur civière | Part des séjours sur civière de 48 h ou plus |

La fonction `part_pct` renvoie **NaN** quand le dénominateur vaut 0, au lieu de provoquer une division par zéro. C'est le cas de Radisson en 2026-2027 (0 visite sur civière), en cohérence avec D4.

⚠️ **Pour agréger ces parts** (par région ou par année), il faut **recalculer la part à partir des sommes des comptes**, par exemple somme des 75 ans et plus / somme des visites. Il ne faut pas faire la moyenne des pourcentages des installations. C'est le même principe que la règle R1.

### 3.2 Variation annuelle des visites par installation

In [16]:
# V8 — Variation des visites d'une installation par rapport à l'année précédente (en %)
# Calculée seulement si l'année précédente existe pour la même installation (années consécutives)
df = df.sort_values(["no_permis_installation", "annee_debut"])
precedente = df.groupby("no_permis_installation")[["annee_debut", "nb_visites_total"]].shift(1)
annees_consecutives = (df["annee_debut"] - precedente["annee_debut"]).eq(1)
df["variation_visites_pct"] = (100 * (df["nb_visites_total"] / precedente["nb_visites_total"] - 1)).where(annees_consecutives)

# Retour à l'ordre de lecture habituel
df = df.sort_values(["annee", "rss", "nom_etablissement", "nom_installation"]).reset_index(drop=True)

print("Lignes sans variation calculable, par année :")
print(df[df["variation_visites_pct"].isna()].groupby("annee").size().to_string())

# Exemple : une installation suivie sur les 6 années
exemple = df.loc[df["perimetre_constant"], "no_permis_installation"].iloc[0]
df.loc[df["no_permis_installation"] == exemple, ["annee", "nom_installation", "nb_visites_total", "variation_visites_pct"]]

Lignes sans variation calculable, par année :
annee
2021-2022    115
2025-2026      1
2026-2027      4


,annee,nom_installation,nb_visites_total,variation_visites_pct
0,2021-2022,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,10418,NaN
115,2022-2023,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,11553,10.894605
230,2023-2024,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,11057,-4.293257
345,2024-2025,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,12646,14.370987
460,2025-2026,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,12034,-4.839475
576,2026-2027,CENTRE HOSPITALIER RÉGIONAL DU GRAND-PORTAGE,12604,4.736580


**Ce qu'on observe.** `variation_visites_pct` (V8) donne l'évolution, en %, des visites d'une installation par rapport à **l'année précédente pour la même installation**, identifiée par son numéro de permis. Exemple ci-dessus : le Grand-Portage passe de 10 418 à 11 553 visites, soit +10,9 %.

Elle n'est **pas calculable** (NaN) pour **120 lignes**, soit la première année de présence de chaque installation :
- 115 en 2021-2022 (première année du fichier) ;
- 1 en 2025-2026 (Chibougamau, première apparition) ;
- 4 en 2026-2027 (les 4 nouvelles installations de Baie-James).

Le calcul exige deux années **consécutives** : une installation qui disparaîtrait puis reviendrait n'aurait pas de variation calculée sur l'intervalle. Ce cas n'existe pas dans le fichier actuel.

### 3.3 Contrôle des variables dérivées

In [17]:
derivees = ["annee_debut"] + parts + ["variation_visites_pct"]

# Toute part ou taux doit être compris entre 0 et 100 %
for col in parts:
    assert df[col].dropna().between(0, 100).all(), f"{col} sort de l'intervalle [0, 100]"
print("Toutes les parts et tous les taux sont entre 0 et 100 %.")

# Les valeurs manquantes doivent correspondre exactement aux cas prévus
attendu_civiere = df["nb_visites_sur_civiere"].eq(0)
assert df["taux_civiere_24h_pct"].isna().equals(attendu_civiere), "NaN inattendu dans taux_civiere_24h_pct"
assert df["taux_civiere_48h_pct"].isna().equals(attendu_civiere), "NaN inattendu dans taux_civiere_48h_pct"
print("NaN par variable dérivée :", df[derivees].isna().sum()[lambda s: s > 0].to_dict())

df[derivees].describe().T.round(2)

Toutes les parts et tous les taux sont entre 0 et 100 %.
NaN par variable dérivée : {'taux_civiere_24h_pct': 1, 'taux_civiere_48h_pct': 1, 'variation_visites_pct': 120}


,count,mean,std,min,25%,50%,75%,max
annee_debut,684.0,2023.48,1.70,2021.00,2022.00,2023.00,2025.00,2026.00
part_75ans_plus_pct,684.0,15.41,5.34,0.00,12.83,15.84,18.54,35.54
part_sante_mentale_pct,684.0,7.23,14.81,1.23,3.10,4.64,6.21,99.81
part_civiere_pct,684.0,27.63,13.22,0.00,18.12,26.97,37.59,65.46
taux_pec_pct,684.0,82.57,8.09,50.17,77.56,83.10,88.33,100.00
taux_civiere_24h_pct,683.0,16.84,13.11,0.00,4.92,14.22,27.02,66.73
taux_civiere_48h_pct,683.0,4.24,5.64,0.00,0.23,1.72,6.11,30.25
variation_visites_pct,564.0,4.97,36.13,-43.29,-2.63,2.42,8.43,676.03


**Ce qu'on observe.**

- Les **6 parts et taux sont tous entre 0 et 100 %** (contrôle automatique).
- Les seuls NaN sont **ceux qui étaient prévus** : 1 pour chacun des deux taux sur civière (Radisson, 0 visite sur civière), vérifié ligne à ligne, et 120 pour la variation (premières années).

**Valeurs extrêmes, toutes expliquées par la nature des installations :**
- `part_sante_mentale_pct` jusqu'à **99,8 %** : l'Institut universitaire en santé mentale de Montréal, une urgence psychiatrique ;
- `part_75ans_plus_pct` = **0 %** : le CHU Sainte-Justine, un hôpital pédiatrique ;
- `taux_pec_pct` = **100 %** et `part_civiere_pct` = **0 %** : Radisson en 2026-2027 (400 visites).

⚠️ **Point ouvert pour l'équipe : trois sauts de volume.** Seules 3 variations dépassent ±50 % ; la suivante en valeur absolue est −43,3 % (Fortierville, 2025-2026) :

| Installation | Région | Passage | Visites | Variation |
|---|---|---|---|---|
| CHSLD et Hôpital Paul-Gilbert | Chaudière-Appalaches | 2021-2022 → 2022-2023 | 3 196 → 24 802 | **+676 %** |
| Centre de santé de Chibougamau | Nord-du-Québec | 2025-2026 → 2026-2027 | 978 → 5 770 | +490 % |
| CMSSS Chauveau | Capitale-Nationale | 2021-2022 → 2022-2023 | 13 019 → 23 559 | +81 % |

Pour Paul-Gilbert et Chauveau, le volume reste ensuite **stable** de 2022-2023 à 2026-2027. Pour Chibougamau, la seule année précédente (2025-2026) est sa toute première année dans le fichier. Dans chaque cas, la première année ressemble donc à une **déclaration partielle** (urgence ouverte, transférée ou déclarée en cours de période) plutôt qu'à une vraie hausse de fréquentation. Le fichier ne permet pas de le confirmer.

Paul-Gilbert et Chauveau font partie du **périmètre constant** : leur année 2021-2022 pèse sur les comparaisons qui démarrent en 2021-2022. Rien n'est modifié ici. **Décision de l'équipe (O1, option (d), 2026-11-11)** : les comparaisons entre années sont présentées avec deux années de départ, 2021-2022 et 2022-2023, à périmètre constant. Cette double présentation sert de contrôle complémentaire. Voir `docs/journal_decisions.md`.

## 4. Contrôle et sauvegarde

### 4.1 Les totaux recalculés égalent-ils le total provincial officiel ?

In [18]:
comptes = [c for c in df.columns if c.startswith("nb_")]
officiel = (df_brut[df_brut["region"].eq("Total provincial")]
            .set_index("annee")[comptes].sort_index())
recalcule = df.groupby("annee")[comptes].sum().sort_index()

ecart_max = (recalcule - officiel).abs().max().max()
print(f"{len(comptes)} colonnes de comptes × {len(officiel)} années : écart absolu maximal = {ecart_max}")
assert ecart_max == 0, "Les totaux recalculés ne correspondent pas au total provincial"

pd.DataFrame({"visites recalculées": recalcule["nb_visites_total"],
              "visites officielles": officiel["nb_visites_total"],
              "écart": recalcule["nb_visites_total"] - officiel["nb_visites_total"]})

14 colonnes de comptes × 6 années : écart absolu maximal = 0


,visites recalculées,visites officielles,écart
annee,,,
2021-2022,1812368,1812368,0
2022-2023,1930100,1930100,0
2023-2024,1884897,1884897,0
2024-2025,2077882,2077882,0
2025-2026,2011715,2011715,0
2026-2027,1858526,1858526,0


**Ce qu'on observe.** Pour les **14 colonnes de comptes** (visites, 75 ans et plus, santé mentale, prises en charge, séjours de plus de 24 h et de plus de 48 h, en version totale, ambulatoire et sur civière) et pour les **6 années**, la somme des installations nettoyées est **exactement égale** au « Total provincial » officiel du fichier brut : l'écart maximal est de 0.

Le nettoyage n'a donc **perdu ni dupliqué aucune visite**. Le contrôle est automatique : l'`assert` arrête le notebook si un écart apparaît.

In [19]:
# Les durées ne s'additionnent pas : on vérifie la moyenne pondérée (règle R1, validée)
poids = {
    "dms_total": "nb_visites_total", "dms_ambulatoire": "nb_visites_ambulatoire",
    "dms_sur_civiere": "nb_visites_sur_civiere", "delai_pec_total": "nb_usagers_pec_total",
    "delai_pec_ambulatoire": "nb_usagers_pec_ambulatoire", "delai_pec_sur_civiere": "nb_usagers_pec_sur_civiere",
}
officiel_durees = df_brut[df_brut["region"].eq("Total provincial")].set_index("annee").sort_index()
lignes = []
for duree, w in poids.items():
    x = df.dropna(subset=[duree])
    ponderee = (x[duree] * x[w]).groupby(x["annee"]).sum() / x[w].groupby(x["annee"]).sum()
    simple = x.groupby("annee")[duree].mean()
    lignes.append({"durée": duree, "poids": w,
                   "écart max pondérée (h)": (ponderee - officiel_durees[duree]).abs().max(),
                   "écart max moyenne simple (h)": (simple - officiel_durees[duree]).abs().max()})
pd.DataFrame(lignes).round(4)

,durée,poids,écart max pondérée (h),écart max moyenne simple (h)
0,dms_total,nb_visites_total,0.0,1.5087
1,dms_ambulatoire,nb_visites_ambulatoire,0.0,0.4989
2,dms_sur_civiere,nb_visites_sur_civiere,0.0,2.6695
3,delai_pec_total,nb_usagers_pec_total,0.0,0.2731
4,delai_pec_ambulatoire,nb_usagers_pec_ambulatoire,0.0,0.2443
5,delai_pec_sur_civiere,nb_usagers_pec_sur_civiere,0.0,0.3743


**Ce qu'on observe.** Les durées sont des moyennes : on ne peut pas les additionner. On vérifie donc qu'une **moyenne pondérée** des installations redonne la valeur provinciale officielle :

- avec les bons poids (le nombre de visites pour les durées de séjour, le nombre d'usagers pris en charge pour les délais), l'écart est **nul** pour les 6 durées et les 6 années ;
- une **moyenne simple** s'en écarte jusqu'à **1,51 h** (`dms_total`) et **2,67 h** (`dms_sur_civiere`).

Ce contrôle confirme que les durées des installations sont intactes. Il justifie aussi la règle **R1** de `docs/journal_decisions.md` (toujours pondérer), **validée** par l'équipe le 2026-11-11.

### 4.2 Sauvegarde dans `data/processed/`

In [20]:
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)
FICHIER_PROPRE = DOSSIER_PROCESSED / "urgences_propre.csv"
df.to_csv(FICHIER_PROPRE, index=False, encoding="utf-8")

# Relecture : le fichier enregistré redonne-t-il exactement le même tableau ?
relu = pd.read_csv(FICHIER_PROPRE, encoding="utf-8", dtype={"no_permis_installation": str})
pd.testing.assert_frame_equal(relu, df, check_dtype=False)
print(f"Enregistré : data/processed/{FICHIER_PROPRE.name} — {relu.shape[0]} lignes × {relu.shape[1]} colonnes, relecture identique")
relu.dtypes.value_counts()

Enregistré : data/processed/urgences_propre.csv — 684 lignes × 36 colonnes, relecture identique


int64      17
float64    13
str         5
bool        1
Name: count, dtype: int64

**Ce qu'on observe.** Le fichier **`data/processed/urgences_propre.csv`** est enregistré : **684 lignes × 36 colonnes**, soit les 27 d'origine, plus `perimetre_constant` et les 8 variables dérivées V1 à V8. Il est en UTF-8 sans BOM. Relu avec pandas, il redonne exactement le même tableau (`assert_frame_equal`).

**Pour le relire dans les autres notebooks :**
```python
df = pd.read_csv(RACINE / "data" / "processed" / "urgences_propre.csv",
                 dtype={"no_permis_installation": str})
```
L'option `dtype` garde le numéro de permis en texte : c'est un identifiant, pas une quantité.

`perimetre_constant` est relu automatiquement comme booléen.